# **PARTE C: RAG**

> Para el desarrollo y la optimización de este Notebook, se ha integrado la capacidad analítica de Gemini 3.1 Pro (la versión avanzada y actual de la familia Gemini) junto con el Modo IA de Google, herramientas que han permitido refinar la lógica del código, asegurar la precisión en el procesamiento de lenguaje natural y validar la coherencia semántica de las respuestas generadas.

Vamos a mejorar las recomendaciones de Gemini utilizando un RAG que haga uso
de las reviews de las películas que hay en iMDB.

* Dados los dos archivos `reviews.csv` y `cleaned_reviews4.csv` que están en el
campus virtual (obtenidos de Kaggle1 ), construye un archivo `.txt` que tenga
la estructura siguiente:
```
    Movie Title: “Titanic”
    Review: “No me gustó la película”
    Review: “qué guay”
    …

    Movie Title: “Avatar”
    Review: “No me gustó la película”
    Review: “qué guay”
    …
```
Ese archivo `.txt` formará la memoria del RAG

### 1. Instalación de los paquetes en el entorno.

Si no tienemos LangChain instalado en nuestro entorno, ejecutamos esta línea:

In [6]:
!pip install langchain langchain-google-genai langchain-openai langchain-text-splitters langchain-experimental

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.7/506.7 kB 29.0 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.19
    Uninstalling langchain-core-1.2.19:
      Successfully uninstalled langchain-core-1.2.19


### 2. Cargando las librerias.

In [1]:
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import google.generativeai as genai
from google.colab import userdata
from google.colab import drive

import pandas as pd
import numpy as np
import os

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


### 3. Creamos acceso a Google Drive para acceder a los archivos `*.csv`

In [2]:
drive.mount('/content/drive')

Mounted at /content/drive


### 4. Descubrir los nombres de las columnas

Si cargamos los csv desde Google Drive.

In [3]:
archivos_csv = [
    '/content/drive/MyDrive/Colab Notebooks/lab4/reviews.csv',
    '/content/drive/MyDrive/Colab Notebooks/lab4/cleaned_reviews4.csv'
]
archivo_salida = '/content/drive/MyDrive/Colab Notebooks/lab4/memoria_rag.txt'

O se carga en local.

In [ ]:
archivos_csv = [
    'reviews.csv',
    'cleaned_reviews4.csv'
]
archivo_salida = 'memoria_rag.txt'

Listamos las columnas para ver la de más peso.

In [4]:
df_prueba = pd.read_csv(archivos_csv[0])
df_prueba2 = pd.read_csv(archivos_csv[-1])
print("Las columnas de reviews.csv son:", list(df_prueba.columns))
print("Las columnas de cleaned_reviews4.csv son:", list(df_prueba2.columns))

Las columnas de reviews.csv son: ['Unnamed: 0', 'position', 'midb_id', 'movie', 'spoilers', 'rating', 'title', 'user', 'date', 'review']
Las columnas de cleaned_reviews4.csv son: ['review_id', 'review', 'movie', 'tconst', 'rating']


### 5. Vamos a construir el archivo .txt

El título de la película está en la columna llamada `movie` (en ambos archivos) y el texto de la reseña está en la columna llamada `review`(también en ambos).

In [5]:

COLUMNA_TITULO = 'movie'
COLUMNA_REVIEW = 'review'

dataframes = []

# 2. Leer y combinar los archivos CSV
for archivo in archivos_csv:
    try:
        print(f"Leyendo {archivo}...")
        df = pd.read_csv(archivo)
        if COLUMNA_TITULO in df.columns and COLUMNA_REVIEW in df.columns:
            # Extraemos solo las dos columnas que nos importan
            dataframes.append(df[[COLUMNA_TITULO, COLUMNA_REVIEW]])
            print(f"✅ Extraídas {len(df)} reseñas de {archivo}")
        else:
            print(f"⚠️ Las columnas no coinciden en {archivo}.")
    except Exception as e:
        print(f"⚠️ Error al abrir {archivo}: {e}")

# 3. Procesar y escribir el .txt
if dataframes:
    print("\nCombinando datos y creando el archivo de memoria...")
    df_final = pd.concat(dataframes, ignore_index=True)
    df_final.dropna(subset=[COLUMNA_TITULO, COLUMNA_REVIEW], inplace=True)

    # Agrupamos todas las reseñas por cada película
    peliculas_agrupadas = df_final.groupby(COLUMNA_TITULO)

    with open(archivo_salida, 'w', encoding='utf-8') as f:
        for titulo, grupo in peliculas_agrupadas:
            f.write(f'Movie Title: "{titulo}"\n')
            for resena in grupo[COLUMNA_REVIEW]:
                # Limpiamos saltos de línea para mantener el formato estricto
                resena_limpia = str(resena).replace('\n', ' ').replace('\r', '').strip()
                f.write(f'Review: "{resena_limpia}"\n')
            f.write('\n')

    print(f"🎉 ¡Completado con éxito! Tienes tu archivo final guardado en:\n{archivo_salida}")

Leyendo /content/drive/MyDrive/Colab Notebooks/lab4/reviews.csv...
✅ Extraídas 20311 reseñas de /content/drive/MyDrive/Colab Notebooks/lab4/reviews.csv
Leyendo /content/drive/MyDrive/Colab Notebooks/lab4/cleaned_reviews4.csv...
✅ Extraídas 261391 reseñas de /content/drive/MyDrive/Colab Notebooks/lab4/cleaned_reviews4.csv

Combinando datos y creando el archivo de memoria...
🎉 ¡Completado con éxito! Tienes tu archivo final guardado en:
/content/drive/MyDrive/Colab Notebooks/lab4/memoria_rag.txt


### 6. Leemos el archivo y lo partimos en Chunks

Leemos el archivo y lo partimos en trozos (chunks). Vamos a poner el
tamaño de cada chunk a 30.000 caracteres.

In [6]:
# 1. Indicamos la ruta de nuestro archivo .txt
archivo_txt = archivo_salida

# 2. Leemos todo el contenido del archivo
try:
    with open(archivo_txt, 'r', encoding='utf-8') as f:
        texto_completo = f.read()
    print(f"✅ Archivo leído correctamente. Tiene un total de {len(texto_completo)} caracteres.")

    # 3. Configuramos el "partidor" de texto
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 30000,
        chunk_overlap = 1000, # solapamiento de fragmentos
        length_function = len,
        separators=["\n\n", "\n", " ", ""]
    )

    # 4. Partimos el texto
    chunks = text_splitter.create_documents([texto_completo])

    # 5. Comprobamos los resultados
    print(f"🍕 ¡Éxito! El texto se ha dividido en {len(chunks)} chunks (trozos).")

    if len(chunks) > 0:
        print(f"📏 Tamaño del primer chunk: {len(chunks[0].page_content)} caracteres.")

except Exception as e:
    print(f"❌ Error: {e}")

✅ Archivo leído correctamente. Tiene un total de 361477042 caracteres.
🍕 ¡Éxito! El texto se ha dividido en 16799 chunks (trozos).
📏 Tamaño del primer chunk: 28578 caracteres.


### 7. Vectorizar texto

Para que el modelo pueda procesar los chunks de texto, hay que vectorizar el texto. Se puede hacer de una forma simple con *TfidfVectorizer* de la librería sklearn [aquí hay un ejemplo]( https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html). Tfidf no crea wordembeddings, no captura relaciones semánticas , solo la frecuencia de las palabras.

Es un detalle técnico crucial que a muchos se les escapa: **TF-IDF (Term Frequency - Inverse Document Frequency)** es un enfoque léxico (basado en palabras exactas) y no semántico. Si un usuario busca "película de miedo" y el texto dice "cine de terror", TF-IDF no sabrá conectarlos porque no entiende el significado, solo cuenta qué palabras se repiten.

Sin embargo, para un entorno académico o un primer prototipo, usar *TfidfVectorizer* es una idea fantástica porque es rapidísimo, consume muy poca memoria y no requiere conectarse a ninguna API externa.

In [7]:
# 1. Extraer el texto puro de los objetos de LangChain
# Usamos una comprensión de listas para sacar el 'page_content' de cada chunk
textos_chunks = [chunk.page_content for chunk in chunks]

print(f"Preparando {len(textos_chunks)} textos para vectorizar...")

# 2. Inicializar el vectorizador TF-IDF
# Truco: Añadimos 'stop_words' para que ignore palabras sin valor como "the", "and", "is"
vectorizador_tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=10000  # Opcional: limitamos a las 10.000 palabras más importantes para ahorrar RAM
)

# 3. Convertir el texto a números (Entrenar y transformar)
# Esto crea una matriz dispersa (sparse matrix) donde cada fila es un chunk y cada columna una palabra
matriz_tfidf = vectorizador_tfidf.fit_transform(textos_chunks)

# 4. Ver los resultados
print("✅ ¡Vectorización completada con éxito!")
print(f"📊 Forma de la matriz TF-IDF: {matriz_tfidf.shape}")
print(f"   -> Filas: {matriz_tfidf.shape[0]} (Tus chunks de texto)")
print(f"   -> Columnas: {matriz_tfidf.shape[1]} (Palabras únicas en el vocabulario)")

# Opcional: Ver algunas de las palabras que ha aprendido el modelo
palabras_ejemplo = vectorizador_tfidf.get_feature_names_out()[:10]
print(f"🔤 Primeras 10 palabras del vocabulario: {palabras_ejemplo}")

Preparando 16799 textos para vectorizar...
✅ ¡Vectorización completada con éxito!
📊 Forma de la matriz TF-IDF: (16799, 10000)
   -> Filas: 16799 (Tus chunks de texto)
   -> Columnas: 10000 (Palabras únicas en el vocabulario)
🔤 Primeras 10 palabras del vocabulario: ['00' '000' '10' '100' '1000' '11' '12' '13' '13th' '14']


### 8. Crea una función ask (que reciba una pregunta)

Crea una función ask (que reciba una pregunta) y sea capaz de devolver
una respuesta en base a las reviews del texto. Pasos:
* Vectorizar la pregunta
* Aplicar una métrica de similitud entre la pregunta y los chunks. Se
puede usar cosine de la librería sklearn.
* Sacar el chunk más similar a la pregunta
* Pasarle al modelo Gemini un prompt que incluya: (1) como contexto
el chunk más similar y (2) la pregunta.

Para el calculo de la distancia entre la pregunta y los textos, se utiliza `cosine_similarity` (similitud del coseno), que mide el ángulo entre dos vectores. Si el ángulo es pequeño (los vectores apuntan en la misma dirección), significa que los textos hablan de lo mismo.

Al diseñar el prompt de esta manera, estámos obligando a Gemini a actuar como un analista de datos. Ya no tira de su memoria de internet, sino que lee exclusivamente las opiniones de los archivos CSV (que extrajimos en ese chunk ganador). Si le preguntamos por una película que no está en el dataset, el coseno de similitud será muy bajo, Gemini leerá un texto que no tiene nada que ver, y gracias a las instrucciones nos dirá: "No tengo suficiente información". ¡Cero alucinaciones!

In [8]:
# 1. Cargar la API Key en el entorno (como vimos en el paso anterior)
try:
    mi_llave = userdata.get('GOOGLE_API_KEY')
    os.environ["GOOGLE_API_KEY"] = mi_llave
except Exception as e:
    print("⚠️ Error al cargar la API Key. Verifica los secretos de Colab.")

# 2. Inicializar el modelo Gemini a través de LangChain
# Ponemos temperature=0 para que sea más analítico y menos "creativo" (ideal para RAG)
modelo = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=os.getenv("GOOGLE_API_KEY"), temperature=0)

# 3. Crear el Template (Plantilla) del Prompt
# LangChain usa las llaves { } para saber dónde inyectar las variables más tarde
template_rag = """
Eres un asistente experto en cine. Tu tarea es responder a la pregunta del usuario basándote ÚNICAMENTE en el contexto proporcionado (que son reseñas reales de usuarios).
Si la respuesta no está en el contexto, di amablemente "No tengo suficiente información en las reseñas para responder a esto" y no inventes datos.

CONTEXTO (Reseñas de películas):
{contexto}

PREGUNTA DEL USUARIO:
{pregunta}
"""
# Convertimos el texto en un objeto de plantilla inteligente de LangChain
prompt_template = ChatPromptTemplate.from_template(template_rag)

# 4. Crear nuestra Cadena (Chain)
# El símbolo | conecta el prompt con el modelo.
# Significa: "Rellena el prompt y pásaselo directamente a Gemini"
cadena_rag = prompt_template | modelo

# 5. La función ask
def ask(pregunta_usuario):
    print(f"🤔 Pregunta: '{pregunta_usuario}'")

    # PASO A: Recuperación (Retrieval) - Igual que antes
    # Utilizamos transform() en lugar de fit_transform() porque el vocabulario ya está entrenado
    pregunta_vectorizada = vectorizador_tfidf.transform([pregunta_usuario])
    similitudes = cosine_similarity(pregunta_vectorizada, matriz_tfidf)

    # 6. Buscamos la POSICIÓN del número más alto
    indice_mejor_chunk = np.argmax(similitudes)
    # 7. (Opcional) Guardamos la puntuación para verla en pantalla
    score_similitud = similitudes[0, indice_mejor_chunk]
    # 8. ¡AQUÍ EXTRAEMOS EL CHUNK!
    # Usamos esa posición para sacar el texto exacto de nuestra lista de chunks
    chunk_mas_similar = textos_chunks[indice_mejor_chunk]

    print(f"🔍 ¡Contexto encontrado! (Similitud: {score_similitud:.4f})")
    print("-" * 50)

    # PASO B: Generación con LangChain
    try:
        # Usamos .invoke() pasándole un diccionario con las variables que definimos en el { } del prompt
        respuesta = cadena_rag.invoke({
            "contexto": chunk_mas_similar,  # (1) Le pasamos el texto ganador del paso anterior
            "pregunta": pregunta_usuario    # (2) Le pasamos lo que el usuario ha preguntado
        })
        # LangChain devuelve un objeto complejo, el texto puro está dentro de .content
        return respuesta.content
    except Exception as e:
        return f"❌ Hubo un error al consultar a LangChain/Gemini: {e}"


### 9. Prueba a ejecutar la función ask(pregunta) 2 veces
Prueba a ejecutar la función ask(pregunta) 2 veces, la primera pasándole como pregunta "What movies receive the most positive reviews?", y la segunda donde la pregunta sea: "What movies receive the most negative reviews?" IMPORTANTE que el prompt esté en inglés, ya que las reviews
están en inglés.
* Analiza los resultados para cada pregunta

In [9]:
def main():
   # Llamada a la función
    print(ask.__doc__)
    pregunta_marta = "What movies receive the most positive reviews?"
    respuesta_final = ask(pregunta_marta)
    print("\n🤖 Respuesta de Gemini (Vía LangChain):")
    print(respuesta_final)
    print("")
    pregunta_marta = "What movies receive the most negative reviews?"
    respuesta_final = ask(pregunta_marta)
    print("\n🤖 Respuesta de Gemini (Vía LangChain):")
    print(respuesta_final)
    print("")

    pregunta_prueba = "¿Qué dicen las reseñas sobre la película Titanic?"
    respuesta_final = ask(pregunta_prueba)
    print("\n🤖 Respuesta de Gemini (Vía LangChain):")
    print(respuesta_final)
    print("")
    pregunta_prueba = "¿Qué dicen las reseñas sobre la película Avatar?"
    respuesta_final = ask(pregunta_prueba)
    print("\n🤖 Respuesta de Gemini (Vía LangChain):")
    print(respuesta_final)
    print("")

if __name__ == "__main__":
    main()

None
🤔 Pregunta: 'What movies receive the most positive reviews?'
🔍 ¡Contexto encontrado! (Similitud: 0.1561)
--------------------------------------------------

🤖 Respuesta de Gemini (Vía LangChain):
No tengo suficiente información en las reseñas para responder a esto. El contexto solo proporciona una reseña detallada de la película "Atithi (1974)" y no compara su recepción con la de otras películas en general para determinar cuáles reciben las críticas más positivas.

🤔 Pregunta: 'What movies receive the most negative reviews?'
🔍 ¡Contexto encontrado! (Similitud: 0.1532)
--------------------------------------------------

🤖 Respuesta de Gemini (Vía LangChain):
No tengo suficiente información en las reseñas para responder a esto.

🤔 Pregunta: '¿Qué dicen las reseñas sobre la película Titanic?'
🔍 ¡Contexto encontrado! (Similitud: 0.4441)
--------------------------------------------------

🤖 Respuesta de Gemini (Vía LangChain):
Las reseñas mencionan la película "Titanic" (refiriéndose p

El sistema ha hecho exactamente lo que le pedimos: buscar un fragmento de texto, leerlo, darse cuenta de que la respuesta global no está ahí, y decir "No lo sé" en lugar de inventarse una mentira (alucinar).

**El límite del Chunk**

Este problema, llamado "Visión de Túnel" ocurre porque el sistema RAG, limitado por el tamaño del chunk (en este caso, 30.000 caracteres), solo analiza un fragmento parcial de la información en lugar de la base de datos completa. Al intentar realizar tareas globales como un ranking, la IA actúa como alguien que debe juzgar toda una biblioteca leyendo solo unas páginas elegidas al azar; por ello, al recuperar únicamente un extracto sobre una película específica (como "Atithi"), el modelo es incapaz de ofrecer una comparativa general y se limita a responder basándose estrictamente en ese contexto reducido.

**El problema del TF-IDF con preguntas analíticas**

La baja puntuación de similitud (aprox. 0.15)
revela las limitaciones del `TfidfVectorizer`, el cual busca coincidencias exactas de palabras en lugar de entender conceptos complejos. Mientras que el RAG tradicional brilla en preguntas específicas sobre un tema concreto, falla en tareas analíticas o de agregación (como ránkings o conteos) porque el sistema no puede procesar la base de datos completa de una vez, sino que recupera fragmentos aislados que contienen las palabras clave pero carecen de la visión global necesaria para responder.

**¿Cómo resolvemos esto en el mundo real?**

Para resolver las limitaciones analíticas en un entorno real, se emplean estrategias que combinan datos estructurados y texto: la
**Estrategia A (Agentes)** permite que la IA ejecute código Python (como Pandas) para obtener cálculos exactos en tiempo real; la **Estrategia B (Metadatos)** consiste en inyectar resúmenes globales y estadísticas clave directamente en el archivo de texto para que el RAG los encuentre fácilmente; y la **Estrategia C** se enfoca en adaptar las consultas a la naturaleza del buscador, priorizando preguntas sobre contenidos específicos. Estas soluciones transforman un simple buscador de fragmentos en un sistema capaz de manejar tanto la trama narrativa como la precisión estadística.

*Estrategia A (Agentes de Datos Estructurados)*

Una de las herramientas más espectaculares que existen ahora mismo en el mundo de la Inteligencia Artificial. Cuando utilizamos un Agente, le estamos dando a Gemini algo más que capacidad de lectura: le estamos dando manos para programar. En lugar de buscar en un texto plano (como hacíamos con el RAG y los chunks), el modelo mira las columnas de CSV, deduce qué código en Python necesita escribir para responder a tu pregunta, lo ejecuta por detrás, lee el resultado y te lo explica.

Gracias al parámetro `verbose=True`, vamos a ver en color verde cómo "piensa" la IA. Veremos algo parecido a esto en nuestra pantalla:

1. Pensamiento de la IA: "Tengo que agrupar por la columna 'movie', calcular la media de 'rating' y ordenar de mayor a menor".
2. Acción: Escribirá un código en Python (tipo `df.groupby('movie')['rating'].mean().sort_values(ascending=False).head()`).
3. Observación: Veremos el resultado de ejecutar ese código.
4. Respuesta Final: Nos dirá con lenguaje natural *"Las películas con mejores valoraciones son X, Y y Z"*.

In [10]:
# 1. Cargamos la API Key (por si acaso)
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

# 2. Leemos tu archivo CSV original directamente
ruta_csv = '/content/drive/MyDrive/Colab Notebooks/lab4/reviews.csv'
df_reviews = pd.read_csv(ruta_csv)

# 3. Inicializamos el modelo Gemini
# temperature=0 es vital aquí para que escriba código exacto y no invente sintaxis
mll = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# 4. Creamos el Agente Analista de Datos
agente = create_pandas_dataframe_agent(
    mll,
    df_reviews,
    verbose=True,              # ¡Magia pura! Esto te mostrará en pantalla el código Python que Gemini escribe
    allow_dangerous_code=True  # LangChain pide esto por seguridad, ya que el agente ejecuta código real
)

# 5. ¡Le hacemos la pregunta analítica que antes falló!
pregunta_marta = "What movies receive the most positive reviews? Please group by movie and check the rating column."

print(f"🤔 Pregunta: '{pregunta_marta}'\n")
print("Empezando el razonamiento del Agente...\n" + "="*50)

try:
    # Usamos invoke para lanzar el agente
    respuesta = agente.invoke(pregunta_marta)

    print("="*50)
    print("\n🤖 Respuesta de Gemini (Agente de Datos):")
    print(respuesta['output'])
except Exception as e:
    print(f"❌ Error al ejecutar el agente: {e}")

🤔 Pregunta: 'What movies receive the most positive reviews? Please group by movie and check the rating column.'

Empezando el razonamiento del Agente...


> Entering new AgentExecutor chain...
Action: python_repl_ast
Action Input:
print(df['rating'].unique())
print(df['rating'].dtype)['1/10' '2/10' '3/10' '4/10' '5/10' '6/10' '7/10' '8/10' '9/10' '10/10']
object
Action: python_repl_ast
Action Input:
df['numeric_rating'] = df['rating'].apply(lambda x: float(x.split('/')[0]))
positive_reviews = df[df['numeric_rating'] >= 7]
most_positive_movies = positive_reviews.groupby('movie').size().sort_values(ascending=False)
print(most_positive_movies.head(10))movie
12 Angry Men              100
Amadeus                   100
Aliens                    100
Avengers: Infinity War    100
Back to the Future        100
American Beauty           100
Interstellar              100
It\'s a Wonderful Life    100
Inglourious Basterds      100
Avengers: Endgame         100
dtype: int64
I now know the final ans

### 10. Ejecuta la misma función 2 veces más
Ejecuta la misma función 2 veces más: la primera vez donde los chunks
tengan tamaño 100.000 caracteres, y la segunda donde los chunks sean de tamaño 200.000 caracteres.

In [11]:
def chunks_y_vectorizar(tamano):
  try:
      # crear chunks
      text_splitter = RecursiveCharacterTextSplitter(
          chunk_size = tamano,
          chunk_overlap = 1000,
          length_function = len,
          separators=["\n\n", "\n", " ", ""]
      )
      chunks = text_splitter.create_documents([texto_completo])

      # vectorización texto
      textos_chunks = [chunk.page_content for chunk in chunks]
      vectorizador_tfidf = TfidfVectorizer(
          stop_words='english',
          max_features=10000
      )
      matriz_tfidf = vectorizador_tfidf.fit_transform(textos_chunks)

      # inicializar modelo
      modelo = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=os.getenv("GOOGLE_API_KEY"), temperature=0)
      template_rag = """
      Eres un asistente experto en cine. Tu tarea es responder a la pregunta del usuario basándote ÚNICAMENTE en el contexto proporcionado (que son reseñas reales de usuarios).
      Si la respuesta no está en el contexto, di amablemente "No tengo suficiente información en las reseñas para responder a esto" y no inventes datos.

      CONTEXTO (Reseñas de películas):
      {contexto}

      PREGUNTA DEL USUARIO:
      {pregunta}
      """
      prompt_template = ChatPromptTemplate.from_template(template_rag)
      cadena_rag = prompt_template | modelo

      return cadena_rag, vectorizador_tfidf, matriz_tfidf
  except Exception as e:
      print(f"❌ Error: {e}")

Ejecuta el main

In [12]:
def main():
    print(chunks_y_vectorizar.__doc__)
    cadena_rag, vectorizador_tfidf, matriz_tfidf = chunks_y_vectorizar(100000)
    print(ask.__doc__)
    pregunta_marta1 = "What movies receive the most positive reviews?"
    respuesta_final = ask(pregunta_marta1)
    print("\n🤖 Respuesta de Gemini (Vía LangChain) para 100000 y pregunta_marta1:")
    print(respuesta_final)
    print("")
    pregunta_marta2 = "What movies receive the most negative reviews?"
    respuesta_final = ask(pregunta_marta2)
    print("\n🤖 Respuesta de Gemini (Vía LangChain) para 100000 y pregunta_marta2:")
    print(respuesta_final)
    print("")

    cadena_rag, vectorizador_tfidf, matriz_tfidf = chunks_y_vectorizar(200000)
    respuesta_final = ask(pregunta_marta1)
    print("\n🤖 Respuesta de Gemini (Vía LangChain) para 200000 y pregunta_marta1:")
    print(respuesta_final)
    print("")
    respuesta_final = ask(pregunta_marta2)
    print("\n🤖 Respuesta de Gemini (Vía LangChain) para 200000 y pregunta_marta2:")
    print(respuesta_final)
    print("")

if __name__ == "__main__":
    main()


None
None
🤔 Pregunta: 'What movies receive the most positive reviews?'
🔍 ¡Contexto encontrado! (Similitud: 0.1561)
--------------------------------------------------

🤖 Respuesta de Gemini (Vía LangChain) para 100000 y pregunta_marta1:
No tengo suficiente información en las reseñas para responder a esto. El contexto solo proporciona una reseña detallada de "Atithi (1974)" y no compara las reseñas positivas de múltiples películas.

🤔 Pregunta: 'What movies receive the most negative reviews?'
🔍 ¡Contexto encontrado! (Similitud: 0.1532)
--------------------------------------------------

🤖 Respuesta de Gemini (Vía LangChain) para 100000 y pregunta_marta2:
No tengo suficiente información en las reseñas para responder a esto. La reseña proporcionada se centra en "Atithi (1974)" y no compara las reseñas negativas de diferentes películas en general.

🤔 Pregunta: 'What movies receive the most positive reviews?'
🔍 ¡Contexto encontrado! (Similitud: 0.1561)
---------------------------------------

* Analiza los resultados para cada pregunta

El análisis demuestra que los chunks de 30.000 caracteres ofrecen la mayor precisión y similitud, mientras que aumentar el tamaño a 100.000 o 200.000 caracteres degrada el rendimiento al incrementar el ruido informativo. Al crecer el chunk, el sistema (TF-IDF) se vuelve menos preciso, provocando que la IA ignore información clave o mezcle reseñas irrelevantes.

* Compara los resultados de las 3 veces que se ha ejecutado la función. ¿Cuál parece ser el mejor tamaño de chunk? ¿Por qué crees que ocurre?

El tamaño de 30.000 caracteres es el más efectivo de los tres, ya que permite a la IA localizar información específica con precisión y alta similitud; sin embargo, en entornos profesionales se prefieren fragmentos aún más pequeños (1.000 a 4.000) para evitar el "ruido" informativo y asegurar que el RAG cumpla su función de filtrar solo lo relevante antes de procesarlo.

El rendimiento decae debido a la dilución del TF-IDF, donde la relevancia de los términos clave se pierde en fragmentos demasiado extensos, y al fenómeno "Lost in the Middle", que provoca que la IA ignore información crítica ubicada en el centro de textos largos. Estos problemas demuestran que los chunks masivos confunden al buscador matemático y saturan la capacidad de atención del modelo, reduciendo drásticamente la precisión de la respuesta.

---

# **PARTE OPCIONAL**

Escogiendo el mejor valor de chunk anterior, pregunta al modelo cosas sobre las películas recomendadas al usuario de la **parte A** anterior (recuerda que las preguntas deberían estar en inglés). Teniendo esto en cuenta, compara los resultados anteriores cambiando:

* La métrica de similitud coseno a otra (por ejemplo, alguna de las que están en el foro de ejercicios de métricas de similitud).
* La temperatura de Gemini.
* Tfidf por otra representación con Word-embeddings (Word2Vec o GloVe
por ejemplo).

(Si el .text con la memoria de las reviews no contiene las películas recomendadas,
haría falta añadirlas copiando y pegando de iMDB, no hace falta las 10 películas,
con un par sería suficiente.)


### 1 (OPCIONAL). Instalar la librería de Embeddings

In [13]:
!pip install sentence-transformers langchain_community

### 2 (OPCIONAL). Preparar el entorno de Jupyter Notebook

In [14]:
from sklearn.metrics.pairwise import pairwise_distances
from langchain_community.document_loaders import TextLoader
from sentence_transformers import SentenceTransformer

### 3 (OPCIONAL). Copiar y pegar las reseñas en inglés de nuestra pelicualas.

Resultados de la **PARTE A**:
```
Cargando los nombres de las películas...

Top 10 recomendaciones (películas nuevas) para el usuario 767:
 1. Wrong Trousers, The (1993) (ID: 169) | Calificación esperada: 5.00
 2. One Flew Over the Cuckoo's Nest (1975) (ID: 357) | Calificación esperada: 4.91
 3. Shawshank Redemption, The (1994) (ID: 64) | Calificación esperada: 4.88
 4. Rear Window (1954) (ID: 603) | Calificación esperada: 4.88
 5. North by Northwest (1959) (ID: 480) | Calificación esperada: 4.87
 6. Mr. Smith Goes to Washington (1939) (ID: 136) | Calificación esperada: 4.87
 7. Usual Suspects, The (1995) (ID: 12) | Calificación esperada: 4.86
 8. Wallace & Gromit: The Best of Aardman Animation (1996) (ID: 114) | Calificación esperada: 4.85
 9. Hoop Dreams (1994) (ID: 48) | Calificación esperada: 4.83
 10. Godfather, The (1972) (ID: 127) | Calificación esperada: 4.83

```

In [15]:
resenas_ingles = """
=========================================
EXTRA REVIEWS FOR OPTIONAL PART
=========================================
Movie: The Wrong Trousers
Rating: 5/5
Review: The Wrong Trousers is a brilliant claymation film featuring Wallace and Gromit. The plot revolves around a mysterious penguin who rents a room and uses a pair of robotic techno-trousers to pull off a diamond heist. It is hilarious and incredibly well animated.

Movie: The Shawshank Redemption
Rating: 5/5
Review: The Shawshank Redemption is an absolute masterpiece. The main plot revolves around Andy Dufresne, a banker who is wrongly convicted of murder and sent to a brutal prison. Over the decades, he maintains his hope and eventually plans a brilliant escape. It is a deeply moving story about friendship, hope, and resilience inside a jail.

Movie: One Flew Over the Cuckoo's Nest
Rating: 5/5
Review: Jack Nicholson's performance as Randle McMurphy is iconic in One Flew Over the Cuckoo's Nest. The movie takes place in a mental institution where the rebellious main character clashes with the oppressive Nurse Ratched. It's a powerful film that questions authority in a psychiatric hospital setting.
"""

# Añadimos al final del archivo
with open(archivo_txt, "a", encoding="utf-8") as f:
    f.write(resenas_ingles)
print("✅ Reseñas en inglés añadidas correctamente.\n")

✅ Reseñas en inglés añadidas correctamente.



### 4 (OPCIONAL). Cargar y partir el texto (mejor chunk size)

In [16]:
loader = TextLoader(archivo_txt, encoding="utf-8")
documento = loader.load()

# Usamos el mejor tamaño según nuestro análisis previo
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 30000,
    chunk_overlap = 1000
)
chunks = text_splitter.split_documents(documento)
textos_chunks = [chunk.page_content for chunk in chunks]
print(f"✅ Texto dividido en {len(textos_chunks)} chunks.\n")

✅ Texto dividido en 16799 chunks.



### 5 (OPCIONAL). Cambio a word embeddings (En lugar de TF-IDF)

In [17]:
print("⏳ Cargando modelo de Word Embeddings y vectorizando... (Puede tardar unos segundos)")
# Cargamos un modelo de embeddings ligero y potente
modelo_embeddings = SentenceTransformer('all-MiniLM-L6-v2')
# Transformamos todos los textos a embeddings semánticos
matriz_embeddings = modelo_embeddings.encode(textos_chunks)
print("✅ ¡Textos convertidos a Embeddings!\n")

⏳ Cargando modelo de Word Embeddings y vectorizando... (Puede tardar unos segundos)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ ¡Textos convertidos a Embeddings!



### 6 (OPCIONAL). Configurar gemini con la nueva temperatura.

In [18]:
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
# Aumentamos la temperatura a 0.8 para ver cómo reacciona
llm_opcional = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.8)

template_rag = """
You are a helpful movie expert. Answer the user's question using ONLY the provided context.
If the answer is not in the context, just say "I don't have enough information".

CONTEXT:
{contexto}

USER QUESTION:
{pregunta}
"""
prompt_template = ChatPromptTemplate.from_template(template_rag)
cadena_rag_opcional = prompt_template | llm_opcional

### 7 (OPCIONAL). Función final con distancia de minkowski.

En el foro las métricas de similitud seleccionada para los dos miembros del grupo son:
* [Distancia de Minkowski, Dynamic Time Warping (DTW)](https://cvdof.ucm.es/moodle/mod/forum/discuss.php?d=52129)
* [SSIM (Structural Similarity Index), PSNR (Peak Signal-to-Noise Ratio)](https://cvdof.ucm.es/moodle/mod/forum/discuss.php?d=52143)

De las métricas mencionadas, la **Distancia de Minkowski** es la única opción válida para comparar vectores de texto en un sistema RAG, ya que el **SSIM** y el **PSNR** se limitan exclusivamente al análisis de calidad en imágenes o vídeo, mientras que el **DTW** está diseñado para series temporales y resultaría computacionalmente ineficiente para vectores estáticos como TF-IDF.

En síntesis ¿Qué es la Distancia de Minkowski? Es una generalización matemática de las distancias. Dependiendo de un parámetro $p$, se transforma en otras distancias conocidas:
* Si $p=1$, es la Distancia de Manhattan.
* Si $p=2$, es la famosa Distancia Euclidiana.

Su fórmula matemática es:
$D(X, Y) = (\sum_{i=1}^{n} |x_i - y_i|^p)^{1/p}$

Para implementar la **Distancia de Minkowski** en nuestra función *ask*, debemos sustituir la métrica de `sklearn` y recordar que, a diferencia del coseno (donde buscamos el valor máximo), aquí el resultado óptimo es el mínimo (el más cercano a cero). Es probable que los resultados en nuestra práctica sobre *The Shawshank Redemption* sean menos precisos que con el Coseno, ya que **Minkowski** se ve penalizada por la diferencia de longitud entre la consulta y el fragmento de texto, mientras que el Coseno prioriza la dirección angular de las palabras, ignorando el tamaño del vector.

In [19]:
def ask_opcional(pregunta_usuario):
    print(f"\n🤔 Pregunta: '{pregunta_usuario}'")

    # A. Vectorizamos la pregunta con Embeddings
    pregunta_vectorizada = modelo_embeddings.encode([pregunta_usuario])

    # B. Calculamos distancia usando MINKOWSKI en lugar de similitud del coseno
    # metric='minkowski' con p=2 equivale a Euclidiana
    distancias = pairwise_distances(pregunta_vectorizada, matriz_embeddings, metric='minkowski', p=2)

    # C. Buscamos la distancia MÍNIMA (np.argmin en lugar de argmax)
    indice_mejor_chunk = np.argmin(distancias)
    mejor_distancia = distancias[0, indice_mejor_chunk]
    chunk_mas_similar = textos_chunks[indice_mejor_chunk]

    print(f"🔍 Contexto recuperado (Distancia Minkowski: {mejor_distancia:.4f})")

    # D. Respuesta de Gemini
    respuesta = cadena_rag_opcional.invoke({
        "contexto": chunk_mas_similar,
        "pregunta": pregunta_usuario
    })

    print(f"🤖 Gemini: {respuesta.content}")
    print("-" * 50)

### 8 (OPCIONAL). Lanzar las preguntas en inglés.


In [20]:
print("🚀 LANZANDO EXPERIMENTOS...")

# Pregunta 1: Para probar la temperatura (El personaje de One Flew Over the Cuckoo's Nest)
ask_opcional("How do the reviewers describe the main character in One Flew Over the Cuckoo's Nest?")

# Pregunta 2: Para probar el poder de los embeddings buscando "prisión" (Shawshank)
ask_opcional("What is the main plot of The Shawshank Redemption?")

# Pregunta 3: Pregunta sobre la comedia de animación (The Wrong Trousers)
ask_opcional("Who rents a room and pulls off a diamond heist in The Wrong Trousers?")

🚀 LANZANDO EXPERIMENTOS...

🤔 Pregunta: 'How do the reviewers describe the main character in One Flew Over the Cuckoo's Nest?'
🔍 Contexto recuperado (Distancia Minkowski: 0.8585)
🤖 Gemini: Reviewers describe the main character, McMurphy, in several ways:

*   "a complete low-life jerk who deserved what happened to him at the end"
*   "trying to avoid jail for statutory rape by pretending to be mentally ill"
*   "totally disrupted the hospital"
*   "the one responsible for the death of the young inmate"
*   "had no redeeming qualities"
*   "so altered that it is maddening to watch if you've read [the book]"
*   "a totally different person than the book Murphy"
*   "replaces the staff as the authority figure"
*   "a complete low-life jerk" (repeated)
*   "a low-life jerk who deserved what happened to him"
*   "a retard in a mental institution"
*   "insighted discord and violence"
*   "selfish, reckless, arrogant character"
*   "deserved the lobotomy"
*   "unresponsive"
*   "ignoring him"

Al ejecutar la práctica, se observa que el uso de Embeddings supera al TF-IDF al capturar el significado semántico en inglés en lugar de palabras sueltas, garantizando la recuperación de texto. La Distancia de Minkowski (donde un valor menor indica mayor cercanía) resulta más sensible a la extensión de los textos que el Coseno. Elevar la temperatura a 0.8 aportará fluidez natural a la respuesta, aunque aumenta el riesgo de alucinaciones si la información no figura en el documento.